# RFM Behavioral Segmentation — Cluster Analysis
**Claro tv+** | 1.92M subscriptions | 24-month synthetic history

This notebook covers:
1. **k-Selection** — Elbow, Silhouette, Davies-Bouldin, Calinski-Harabasz across k = 3–7
2. **Cluster Scatter** — PCA (+ UMAP if installed) projection of the 6 segments
3. **Segment Profile Heatmap** — Median R, F, M, coupon usage, NPS per segment
4. **90-Day Churn Curves** — Cumulative cancellation rate per segment

> The full ML pipeline lives in `src/segmentacao_crm.py`. This notebook loads its output and produces the visualizations.


## 1. Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score,
                             davies_bouldin_score,
                             calinski_harabasz_score,
                             adjusted_rand_score)
from sklearn.utils import resample

warnings.filterwarnings('ignore')

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed — PCA only.  pip install umap-learn")

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 130
RANDOM_STATE = 42

SEG_COLORS = {
    'Cinéfilos de cupom':         '#2E75B6',
    'Fiéis a preço cheio':        '#70AD47',
    'Esfriando (alto valor)':     '#ED7D31',
    'Novatos em experimentação':  '#FFC000',
    'Presenteados':               '#A5A5A5',
    'Quase perdidos':             '#C00000',
}


## 2. Load Data

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'clientes_ativos_segmentados.csv.gz')
df = pd.read_csv(DATA_PATH, compression='gzip', low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
df.head(3)


In [ ]:
SEG_COL = 'segmento'
if SEG_COL not in df.columns:
    candidates = [c for c in df.columns if 'seg' in c.lower() or 'cluster' in c.lower()]
    print(f"Column '{SEG_COL}' not found. Candidates: {candidates}")
else:
    print(df[SEG_COL].value_counts().to_string())


## 3. Feature Engineering

In [ ]:
FEATURES     = ['R', 'F', 'M', 'cupom', 'presente', 'nps']
LOG_FEATURES = ['R', 'F', 'M']

missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns: {missing}. Available: {list(df.columns)}")

# FIX: reset_index keeps X_scaled row positions aligned with df_feat row positions
df_feat  = df.dropna(subset=FEATURES).reset_index(drop=True)
X_raw    = df_feat[FEATURES].copy()
X_log    = X_raw.copy()
for col in LOG_FEATURES:
    X_log[col] = np.log1p(X_log[col])

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
print(f"Feature matrix: {X_scaled.shape[0]:,} × {X_scaled.shape[1]}")
if len(df_feat) < len(df):
    print(f"  ({len(df) - len(df_feat):,} rows dropped due to NAs in features)")


## 4. k-Selection Analysis
Running KMeans k = 3–7 on a 150k sample — same protocol as `segmentacao_crm.py`.


In [ ]:
SAMPLE_SIZE = 150_000
K_RANGE     = range(3, 8)

idx_s = resample(np.arange(len(X_scaled)),
                 n_samples=min(SAMPLE_SIZE, len(X_scaled)),
                 random_state=RANDOM_STATE, replace=False)
X_s = X_scaled[idx_s]

rows = []
for k in K_RANGE:
    km     = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_s)
    rows.append({
        'k':                 k,
        'inertia':           km.inertia_,
        'silhouette':        silhouette_score(X_s, labels,
                                              sample_size=10_000,
                                              random_state=RANDOM_STATE),
        'davies_bouldin':    davies_bouldin_score(X_s, labels),
        'calinski_harabasz': calinski_harabasz_score(X_s, labels),
    })
    print(f"  k={k} done")

df_k = pd.DataFrame(rows)
df_k.round(4)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('k-Selection Metrics — RFM+NPS Features  (sample n=150k)',
             fontsize=14, fontweight='bold', y=1.01)

specs = [
    ('inertia',           'Inertia (Elbow)',         '↓ look for the kink', '#2E75B6'),
    ('silhouette',        'Silhouette Score',         '↑ higher is better',  '#70AD47'),
    ('davies_bouldin',    'Davies-Bouldin Score',     '↓ lower is better',   '#ED7D31'),
    ('calinski_harabasz', 'Calinski-Harabasz Score',  '↑ higher is better',  '#A5A5A5'),
]

for ax, (col, title, note, color) in zip(axes.flat, specs):
    ax.plot(df_k['k'], df_k[col], 'o-', color=color, lw=2.5, ms=7)
    ax.axvline(6, color='#C00000', ls='--', lw=1.4, alpha=0.7, label='k = 6 chosen')
    ax.set_title(f'{title}\n{note}', fontsize=11)
    ax.set_xlabel('k'); ax.set_xticks(list(K_RANGE))
    ax.legend(fontsize=9)

plt.tight_layout()
os.makedirs('../docs', exist_ok=True)
plt.savefig('../docs/k_selection_metrics.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved → docs/k_selection_metrics.png")


### 4.1 Stability — Adjusted Rand Index across seeds
ARI ≈ 1 means two runs assign the same clusters; ARI ≈ 0 means random.
The script reports ARI ≈ 0.947 for k = 6.


In [ ]:
N_SEEDS  = 5
ari_rows = []
for k in K_RANGE:
    labels_ref = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_s)
    aris = [adjusted_rand_score(labels_ref,
                KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X_s))
            for seed in range(1, N_SEEDS + 1)]
    ari_rows.append({'k': k, 'ARI_mean': np.mean(aris), 'ARI_min': np.min(aris)})

df_ari = pd.DataFrame(ari_rows)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_ari['k'], df_ari['ARI_mean'], color='#2E75B6', alpha=0.8, label='Mean ARI')
ax.bar(df_ari['k'], df_ari['ARI_min'],  color='#ED7D31', alpha=0.6, label='Min ARI')
ax.axvline(6, color='#C00000', ls='--', lw=1.4, label='k = 6')
ax.set_xlabel('k'); ax.set_ylabel('ARI (0–1)'); ax.set_ylim(0, 1.05)
ax.set_title('Cluster Stability — ARI across seeds', fontweight='bold')
ax.legend(); ax.set_xticks(list(K_RANGE))
plt.tight_layout()
plt.savefig('../docs/ari_stability.png', bbox_inches='tight', dpi=150)
plt.show()
print(df_ari.round(3).to_string(index=False))


## 5. Cluster Scatter — PCA

In [ ]:
PLOT_N = 50_000
idx_p  = resample(np.arange(len(X_scaled)),
                  n_samples=min(PLOT_N, len(X_scaled)),
                  random_state=RANDOM_STATE, replace=False)
X_p    = X_scaled[idx_p]
# FIX: use df_feat (index-aligned with X_scaled) instead of df
segs_p = df_feat.iloc[idx_p][SEG_COL].values

pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_p)
var   = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(11, 7))
for seg, color in SEG_COLORS.items():
    mask = segs_p == seg
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=seg, alpha=0.35, s=7, rasterized=True)

ax.set_title(
    f'PCA Projection — 6 Segments  (n = {PLOT_N:,}, '
    f'explained variance = {var.sum():.1%})',
    fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({var[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({var[1]:.1%})', fontsize=11)
ax.legend(markerscale=4, framealpha=0.92, fontsize=10)
plt.tight_layout()
plt.savefig('../docs/pca_scatter.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved → docs/pca_scatter.png")


## 6. Cluster Scatter — UMAP *(optional)*

In [ ]:
if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                        random_state=RANDOM_STATE, verbose=False)
    X_umap  = reducer.fit_transform(X_p)

    fig, ax = plt.subplots(figsize=(11, 7))
    for seg, color in SEG_COLORS.items():
        mask = segs_p == seg
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   c=color, label=seg, alpha=0.35, s=7, rasterized=True)
    ax.set_title(f'UMAP Projection — 6 Segments  (n = {PLOT_N:,})',
                 fontsize=13, fontweight='bold')
    ax.legend(markerscale=4, framealpha=0.92, fontsize=10)
    plt.tight_layout()
    plt.savefig('../docs/umap_scatter.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → docs/umap_scatter.png")
else:
    print("Skipped — install umap-learn to generate UMAP plot.")


## 7. Segment Profile Heatmap
Each cell shows the **raw median** value; colour encodes the **normalised** value (0 = lowest median, 1 = highest).


In [ ]:
PROFILE_COLS = ['R', 'F', 'M', 'cupom', 'nps']
agg      = df.groupby(SEG_COL)[PROFILE_COLS].median()
agg_norm = (agg - agg.min()) / (agg.max() - agg.min())

if 'churn90' in df.columns:
    agg['churn90 (%)']      = df.groupby(SEG_COL)['churn90'].mean() * 100
    agg_norm['churn90 (%)'] = (agg['churn90 (%)'] - agg['churn90 (%)'].min()) /                               (agg['churn90 (%)'].max() - agg['churn90 (%)'].min())

seg_order = agg['M'].sort_values(ascending=False).index
agg       = agg.loc[seg_order]
agg_norm  = agg_norm.loc[seg_order]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    agg_norm.T,
    annot=agg.T.round(1),
    fmt='.1f',
    cmap='RdYlGn_r',
    linewidths=0.6,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Normalised (0–1)', 'shrink': 0.8}
)
ax.set_title('Segment Profile — Median values (raw) · Colour = normalised rank',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel(''); ax.set_ylabel('')
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('../docs/segment_profile_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved → docs/segment_profile_heatmap.png")


## 8. 90-Day Churn Curves
Cumulative cancellation rate per segment over the 90-day observation window.
Falls back to a bar chart if `dias_ate_cancel` is absent.


In [ ]:
if 'dias_ate_cancel' in df.columns and 'churn90' in df.columns:
    days = np.arange(0, 91)
    fig, ax = plt.subplots(figsize=(12, 6))

    for seg, color in SEG_COLORS.items():
        seg_df   = df[df[SEG_COL] == seg]
        n_total  = len(seg_df)
        churners = seg_df[seg_df['churn90'] == 1]['dias_ate_cancel'].values
        cum_pct  = np.array([(churners <= d).sum() / n_total * 100 for d in days])
        ax.plot(days, cum_pct, lw=2.2, color=color,
                label=f'{seg}  ({cum_pct[-1]:.1f}%)')

    ax.set_title('Churn acumulado 90 dias por segmento', fontsize=13, fontweight='bold')
    ax.set_xlabel('Dias após T0', fontsize=11)
    ax.set_ylabel('% da base do segmento cancelada', fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(loc='upper left', framealpha=0.92, fontsize=10)
    ax.set_xlim(0, 90)
    plt.tight_layout()
    plt.savefig('../docs/churn_curves_90d.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → docs/churn_curves_90d.png")

elif 'churn90' in df.columns:
    # FIX: parentheses around the full expression before .sort_values()
    churn_by_seg = (df.groupby(SEG_COL)['churn90'].mean() * 100).sort_values(ascending=False)
    churn_by_seg = churn_by_seg.loc[[s for s in SEG_COLORS if s in churn_by_seg.index]]

    fig, ax = plt.subplots(figsize=(10, 5))
    colors  = [SEG_COLORS[s] for s in churn_by_seg.index]
    bars    = ax.bar(churn_by_seg.index, churn_by_seg.values, color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=11)
    ax.set_title('Taxa de churn 90 dias por segmento', fontsize=13, fontweight='bold')
    ax.set_ylabel('Churn rate (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    plt.xticks(rotation=25, ha='right', fontsize=10)
    plt.tight_layout()
    plt.savefig('../docs/churn_rate_by_segment.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → docs/churn_rate_by_segment.png")

else:
    print("Neither 'churn90' nor 'dias_ate_cancel' found in the CSV — skipping churn plot.")


## 9. Segment Summary Table

In [ ]:
if 'churn90' in df.columns and 'M' in df.columns:
    agg_dict = {
        'n':         pd.NamedAgg('R',       'count'),
        'rev_share': pd.NamedAgg('M',       'sum'),
        'churn_90d': pd.NamedAgg('churn90', 'mean'),
        'med_R':     pd.NamedAgg('R',       'median'),
        'med_F':     pd.NamedAgg('F',       'median'),
        'med_M':     pd.NamedAgg('M',       'median'),
        'nps_med':   pd.NamedAgg('nps',     'median'),
    }
    smry = df.groupby(SEG_COL).agg(**agg_dict)
    total_rev = smry['rev_share'].sum()
    smry['rev_share_%'] = (smry['rev_share'] / total_rev * 100).round(1)
    smry['churn_90d_%'] = (smry['churn_90d'] * 100).round(1)
    smry['base_%']      = (smry['n'] / smry['n'].sum() * 100).round(1)
    out = smry[['n', 'base_%', 'rev_share_%', 'churn_90d_%', 'med_R', 'med_F', 'med_M', 'nps_med']]
    out = out.sort_values('rev_share_%', ascending=False)
    out.columns = ['n', '% base', '% receita', 'churn 90d %', 'medR', 'medF', 'medM', 'medNPS']
    display(out.style
               .background_gradient(cmap='RdYlGn_r', subset=['churn 90d %'])
               .background_gradient(cmap='Blues',    subset=['% receita'])
               .format({'n': '{:,.0f}', '% base': '{:.1f}%', '% receita': '{:.1f}%',
                        'churn 90d %': '{:.1f}%', 'medR': '{:.0f}', 'medF': '{:.1f}',
                        'medM': '{:.2f}', 'medNPS': '{:.0f}'}))
